In [1]:
from openadmet.toolkit.database.chembl import PermissiveChEMBLTargetCurator
from openadmet.toolkit.chemoinformatics.rdkit_funcs import canonical_smiles, smiles_to_inchikey
from tqdm.auto import tqdm
tqdm.pandas()
import datamol as dm
import os
import subprocess

/Users/cynthiaxu/miniforge3/envs/openadmet-toolkit/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def gather_chembl_data_for_target(target_name: str, chembl_tid: str, chembl_ver: int):
    print(f"working on target {target_name}")
    pctc = PermissiveChEMBLTargetCurator(chembl_target_id=chembl_tid, version=chembl_ver, standard_type="EC50", require_pchembl=True)
    activity_data = pctc.get_activity_data(return_as="df")

    with dm.without_rdkit_log():
        activity_data["OPENADMET_CANONICAL_SMILES"] = activity_data["canonical_smiles"].progress_apply(lambda x: canonical_smiles(x))
        activity_data["OPENADMET_INCHIKEY"] = activity_data["OPENADMET_CANONICAL_SMILES"].progress_apply(lambda x: smiles_to_inchikey(x))
    
    # important to canonicalise here so compound deduplication is done correctly
    aggregated_activity = pctc.aggregate_activity_data_by_compound(canonicalise=True)
    print("smiles duplicates", aggregated_activity["OPENADMET_CANONICAL_SMILES"].duplicated().sum())
    print("inchikey duplicates", aggregated_activity["OPENADMET_INCHIKEY"].duplicated().sum())
    return aggregated_activity, activity_data

In [3]:
targets = {
    "AHR": "CHEMBL3201",
    "PXR": "CHEMBL3401",
}

In [4]:
chembl_ver = 37

In [5]:
from openadmet.toolkit.webservices.credentials import S3Settings
from openadmet.toolkit.webservices.s3 import S3Bucket

In [6]:
os.environ["AWS_ACCESS_KEY_ID"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_access_key_id"],
    text=True
).strip()

os.environ["AWS_SECRET_ACCESS_KEY"] = subprocess.check_output(
    ["aws", "configure", "get", "aws_secret_access_key"],
    text=True
).strip()

In [7]:
settings = S3Settings()

In [8]:
bucket = "openadmet-data-public-dev"

In [9]:
bucket = S3Bucket.from_settings(settings, bucket)

In [10]:
import datetime

In [11]:
t = datetime.datetime.now()

In [12]:
date = t.strftime("%Y-%m-%d")

In [13]:
location=f"ChEMBL{chembl_ver}_EC50"

In [14]:
import os
from pathlib import Path

location_path = Path(location)

In [15]:
location_path.mkdir(exist_ok=False)

In [16]:
uris_raw = {}
uris_agg = {}
for target, chembl_tid in targets.items():

    agg, raw  = gather_chembl_data_for_target(target, chembl_tid, chembl_ver)
    # TODO: make a function this is clunky
    fname_agg = f"ChEMBL_EC50_{target}_{chembl_tid}_aggregated.parquet"
    fname_raw = f"ChEMBL_EC50_{target}_{chembl_tid}_raw.parquet"
    
    agg.reset_index(drop=True).to_parquet(location_path/fname_agg, index=False)
    raw.reset_index(drop=True).to_parquet(location_path/fname_raw, index=False)
    
    bucket_destination_agg = location + "/" + fname_agg
    bucket.push_file(location_path/fname_agg, bucket_destination_agg)
    bucket_destination_raw = location + "/" + fname_raw
    bucket.push_file(location_path/fname_raw, bucket_destination_raw)

    # get S3 URIs
    uri_agg = bucket.to_uri(bucket_destination_agg)
    uris_agg[target] = uri_agg

    uri_raw = bucket.to_uri(bucket_destination_raw)
    uris_raw[target] = uri_raw

working on target AHR


100%|██████████| 276/276 [00:00<00:00, 6691.26it/s]


smiles duplicates 0
inchikey duplicates 0
working on target PXR


100%|██████████| 743/743 [00:00<00:00, 4672.95it/s]


smiles duplicates 0
inchikey duplicates 0


In [17]:
import intake
intake.Catalog?
cat = intake.entry.Catalog()

Init signature:
intake.Catalog(
    entries: 'Iterable[ReaderDescription] | Mapping | None' = None,
    aliases: 'dict[str, int] | None' = None,
    data: 'Iterable[DataDescription] | Mapping' = None,
    user_parameters: 'dict[str, BaseUserParameter] | None' = None,
    parameter_overrides: 'dict[str, Any] | None' = None,
    metadata: 'dict | None' = None,
)
Docstring:      A collection of data and reader descriptions.
File:           ~/miniforge3/envs/openadmet-toolkit/lib/python3.12/site-packages/intake/readers/entry.py
Type:           type
Subclasses:     THREDDSCatalog

In [18]:
uris_agg

{'AHR': 's3://openadmet-data-public-dev/ChEMBL37_EC50/ChEMBL_EC50_AHR_CHEMBL3201_aggregated.parquet',
 'PXR': 's3://openadmet-data-public-dev/ChEMBL37_EC50/ChEMBL_EC50_PXR_CHEMBL3401_aggregated.parquet'}

In [19]:
uris_raw

{'AHR': 's3://openadmet-data-public-dev/ChEMBL37_EC50/ChEMBL_EC50_AHR_CHEMBL3201_raw.parquet',
 'PXR': 's3://openadmet-data-public-dev/ChEMBL37_EC50/ChEMBL_EC50_PXR_CHEMBL3401_raw.parquet'}

In [20]:
for k,v in uris_agg.items():
    cat[k+"_aggregated"] = intake.readers.PandasParquet(v)

In [21]:
for k,v in uris_raw.items():
    cat[k+"_raw"] = intake.readers.PandasParquet(v)

In [22]:
catname = f"CATALOG_{location}.yaml"

In [23]:
cat.to_yaml_file(catname)

In [24]:
cat_location = location+ "/" +catname

In [25]:
cat_location

'ChEMBL37_EC50/CATALOG_ChEMBL37_EC50.yaml'

In [26]:
bucket.push_file(catname, cat_location)

In [27]:
cat_uri = bucket.to_uri(cat_location)